In [1]:
import bw2data, bw2io, bw2calc
import numpy as np
import os
import re
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 120)

In [2]:
wind  = pd.read_excel("../../dp-LCI_output/staticLCI_(p)GWP20/wind_CN_US_staticLCI_threeGWP20.xlsx")
hydro = pd.read_excel("../../dp-LCI_output/staticLCI_(p)GWP20/hydro_reservoir_CN-asRoW_US-asQC_staticLCI_threeGWP20.xlsx")
pv    = pd.read_excel("../../dp-LCI_output/staticLCI_(p)GWP20/PV_CN_US_staticLCI_threeGWP20.xlsx")

In [5]:
def add_diff_columns(df):
    df = df.copy()

    df["diff% pGWP_fixedCO2"] = (
        (df["pGWP20_fixedCO2"] - df["gwp20"]) / df["gwp20"] * 100
    ).round(0).astype(int)

    df["diff% pGWP_dpCO2"] = (
        (df["pGWP20_dpCO2"] - df["gwp20"]) / df["gwp20"] * 100
    ).round(0).astype(int)

    return df


In [6]:
hydro = add_diff_columns(hydro)
wind  = add_diff_columns(wind)
pv    = add_diff_columns(pv)

In [7]:
def highlight_threshold(val):
    if pd.isna(val):
        return ""
    if abs(val) >= 10:
        return "background-color: #ff6b6b; color: black;"   # red
    elif abs(val) >= 5:
        return "background-color: #f4b084; color: black;"   # yellow
    else:
        return ""



In [8]:

hydro.style.applymap(
        highlight_threshold,
        subset=["diff% pGWP_fixedCO2", "diff% pGWP_dpCO2"]
    ).set_properties(
        subset=["Activity"],
        **{"white-space": "nowrap"}
    )


/var/folders/9p/gvkl7h6551515cs56m93ldgr0000gn/T/ipykernel_3555/4053345779.py:1: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  hydro.style.applymap(


,Activity,gwp20,pGWP20_fixedCO2,pGWP20_dpCO2,diff% pGWP_fixedCO2,diff% pGWP_dpCO2
0,"market for electricity, hydro, high voltage, US, SSP1-VLLO, 2030",0.042688,0.048139,0.041953,13,-2
1,"market for electricity, hydro, high voltage, US, SSP1-VLLO, 2050",0.040573,0.047673,0.041773,17,3
2,"market for electricity, hydro, high voltage, US, SSP2-M, 2030",0.042714,0.046155,0.041175,8,-4
3,"market for electricity, hydro, high voltage, US, SSP2-M, 2050",0.042399,0.043565,0.043725,3,3
4,"market for electricity, hydro, high voltage, US, SSP5-H, 2030",0.042774,0.044570,0.040436,4,-5
5,"market for electricity, hydro, high voltage, US, SSP5-H, 2050",0.042535,0.038687,0.042779,-9,1
6,"market for electricity, hydro, high voltage, CN, SSP1-VLLO, 2030",0.061746,0.070191,0.061253,14,-1
7,"market for electricity, hydro, high voltage, CN, SSP1-VLLO, 2050",0.058479,0.067672,0.059373,16,2
8,"market for electricity, hydro, high voltage, CN, SSP2-M, 2030",0.061851,0.068054,0.060778,10,-2
9,"market for electricity, hydro, high voltage, CN, SSP2-M, 2050",0.060988,0.061723,0.061987,1,2


In [9]:
# wind and PV has 2040 rows, and we don't use them 
wind = wind[wind["Activity"].str.contains(r"\b(2030|2050)\b", regex=True)]

wind.style.applymap(
        highlight_threshold,
        subset=["diff% pGWP_fixedCO2", "diff% pGWP_dpCO2"]
    ).set_properties(
        subset=["Activity"],
        **{"white-space": "nowrap"}
    )


/var/folders/9p/gvkl7h6551515cs56m93ldgr0000gn/T/ipykernel_3555/915099166.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  wind = wind[wind["Activity"].str.contains(r"\b(2030|2050)\b", regex=True)]
/var/folders/9p/gvkl7h6551515cs56m93ldgr0000gn/T/ipykernel_3555/915099166.py:4: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  wind.style.applymap(


,Activity,gwp20,pGWP20_fixedCO2,pGWP20_dpCO2,diff% pGWP_fixedCO2,diff% pGWP_dpCO2
0,"market for electricity, wind, high voltage, CN-NM, SSP1-VLLO, 2030",0.040977,0.048007,0.042131,17,3
1,"market for electricity, wind, high voltage, CN-NM, SSP1-VLLO, 2050",0.020379,0.024406,0.021573,20,6
2,"market for electricity, wind, high voltage, CN-NM, SSP2-M, 2030",0.041837,0.047651,0.042756,14,2
3,"market for electricity, wind, high voltage, CN-NM, SSP2-M, 2050",0.034676,0.036135,0.036290,4,5
4,"market for electricity, wind, high voltage, CN-NM, SSP5-H, 2030",0.042915,0.047801,0.043581,11,2
5,"market for electricity, wind, high voltage, CN-NM, SSP5-H, 2050",0.038172,0.035892,0.039536,-6,4
6,"market for electricity, wind, high voltage, US-TRE, SSP1-VLLO, 2030",0.022915,0.026846,0.023560,17,3
7,"market for electricity, wind, high voltage, US-TRE, SSP1-VLLO, 2050",0.011396,0.013648,0.012064,20,6
8,"market for electricity, wind, high voltage, US-TRE, SSP2-M, 2030",0.023395,0.026647,0.023909,14,2
9,"market for electricity, wind, high voltage, US-TRE, SSP2-M, 2050",0.019391,0.020207,0.020293,4,5


In [10]:
pv = pv[pv["Activity"].str.contains(r"\b(2030|2050)\b", regex=True)]

pv.style.applymap(
        highlight_threshold,
        subset=["diff% pGWP_fixedCO2", "diff% pGWP_dpCO2"]
    ).set_properties(
        subset=["Activity"],
        **{"white-space": "nowrap"}
    )


/var/folders/9p/gvkl7h6551515cs56m93ldgr0000gn/T/ipykernel_3555/3604744248.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  pv = pv[pv["Activity"].str.contains(r"\b(2030|2050)\b", regex=True)]
/var/folders/9p/gvkl7h6551515cs56m93ldgr0000gn/T/ipykernel_3555/3604744248.py:3: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  pv.style.applymap(


,Activity,gwp20,pGWP20_fixedCO2,pGWP20_dpCO2,diff% pGWP_fixedCO2,diff% pGWP_dpCO2
0,"market for electricity, PV, low voltage, CN, SSP1-VLLO, 2030",0.035883,0.040738,0.035691,14,-1
1,"market for electricity, PV, low voltage, CN, SSP1-VLLO, 2050",0.011438,0.013117,0.011608,15,1
2,"market for electricity, PV, low voltage, CN, SSP2-M, 2030",0.036803,0.040630,0.036404,10,-1
3,"market for electricity, PV, low voltage, CN, SSP2-M, 2050",0.022702,0.022873,0.022971,1,1
4,"market for electricity, PV, low voltage, CN, SSP5-H, 2030",0.038557,0.041649,0.037925,8,-2
5,"market for electricity, PV, low voltage, CN, SSP5-H, 2050",0.027646,0.025178,0.027763,-9,0
6,"market for electricity, PV, low voltage, US, SSP1-VLLO, 2030",0.022504,0.025587,0.022418,14,0
7,"market for electricity, PV, low voltage, US, SSP1-VLLO, 2050",0.007433,0.008556,0.007571,15,2
8,"market for electricity, PV, low voltage, US, SSP2-M, 2030",0.023150,0.025596,0.022934,11,-1
9,"market for electricity, PV, low voltage, US, SSP2-M, 2050",0.014592,0.014729,0.014793,1,1
